# Notebook pour le Remplissage des NA

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import NotFittedError

## Fonction pour le nettoyage du dataframe

In [2]:
def clean_df(df: pd.DataFrame):
    # On supprime les colonnes ayant plus de 80% de valeurs manquantes
    valeurs_manquantes = df.isna().sum()/df.shape[0]
    valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)
    df_filtered = df.drop(valeurs_manquantes_list, axis = 1)
    # On supprime les colonnes "n6" pour ne laisser que les "n7" qui paraissent plus judicieuses (basées sur un plus grand nombre de bien)
    column_n6 = [ column for column in df_filtered.columns if "n6" in column]
    df_filtered = df_filtered.drop(column_n6, axis = 1)
    if 'typedebien' in df_filtered.columns:
        df_filtered = df_filtered[df_filtered['typedebien'] != 'l']
        df_filtered['typedebien'] = df_filtered['typedebien'].replace({'an': 'a', 'mn': 'm'})

    # On transform en Int des colonnes déclarées en float mais n'ayant que des int
    valeurs_manq_resid_quanti = [col for  col in df_filtered.select_dtypes(exclude='object').columns if df_filtered[col].isna().sum() > 0]    
    col_float = df_filtered[valeurs_manq_resid_quanti].select_dtypes(include='float64').columns
    for col in col_float:
        array_col = np.array(df_filtered[df_filtered[col].notna()][col]) 
        array_col_round = np.round(array_col)
        array_real_float = array_col[array_col != array_col_round]
        if len(array_real_float) == 0:
            df_filtered[col] = df_filtered[col].astype('Int64')
    # on transforme les colonnes quali n'ayant en fait que deux modalités
    if 'cave' in df_filtered.columns:
        df_filtered["cave"] = df_filtered["cave"].astype('Int64')
    if 'ascenseur' in df_filtered.columns:
        df_filtered["ascenseur"] = df_filtered["ascenseur"].astype('Int64')
    if 'logement_neuf' in df_filtered.columns:
        df_filtered["logement_neuf"] = df_filtered["logement_neuf"].replace({'n': False, 'o': True}).astype('Int64')

    return df_filtered

## Transformer pour le remplissage (quali et quanti) du dataframe

In [3]:
class KNNImputerCustom(BaseEstimator, TransformerMixin):
    def __init__(self, type_col="typedebien", lat_col="mapCoordonneesLatitude", lon_col="mapCoordonneesLongitude", pieces_col="nb_pieces", k=10):
        self.type_col = type_col
        self.lat_col = lat_col
        self.lon_col = lon_col
        self.pieces_col = pieces_col
        self.k = k
        self.fitted = False
 
    def fit(self, X, y=None):
        self.df_train_ = X.copy()
        self.numeric_cols_ = X.select_dtypes(exclude="object").columns.tolist()
        self.default_mean_ = {col: X[col].mean() for col in self.numeric_cols_}
        self.imputer_ = KNNImputer()
        self.quali_cols_ = X.select_dtypes(include="object").columns.tolist()
        self.default_mode_ = {col: X[col].mode()[0] for col in self.quali_cols_}
        self.fitted = True
        
        return self
 
    def transform(self, X):
        if not(self.fitted):
            raise NotFittedError("This KNNImputerCustomed instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.")
        X = X.copy()
        # First we fill the numeric columns
        for col in self.numeric_cols_:
            if X[col].isna().sum() == 0:
                continue
            cols = [col, self.pieces_col, self.lat_col, self.lon_col]
            df_extract = X[[self.type_col] + cols]
            df_extract_train = self.df_train_ [[self.type_col] + cols]
            for typ in ["a", "m"]:
                subset = df_extract[df_extract[self.type_col] == typ][cols]
                if subset.shape[0] == 0:
                    continue
                apply_default = True
                subset_train = df_extract_train[df_extract_train[self.type_col] == typ][cols]
                if subset_train.shape[0] != 0:
                    self.imputer_.fit(subset_train)
                    imputed = self.imputer_.transform(subset)
                    if imputed.shape[1] == len(cols):
                        imputed_df = pd.DataFrame(imputed, columns=cols, index=subset.index)
                        apply_default = False
                if apply_default:
                    if X[col].dtype in ["int64", "Int64"]:
                        X.loc[subset.index, col] = self.default_mean_[col].astype("int64")
                    else:
                        X.loc[subset.index, col] = self.default_mean_[col]
                else:
                    if X[col].dtype in ["int64", "Int64"]:
                        X.loc[imputed_df.index, col] = imputed_df[col].round().astype("int64")
                    else:
                        X.loc[imputed_df.index, col] = imputed_df[col]

        #Secondly we fill the qualitative columns
        index_na = X[X[self.quali_cols_].isna().any(axis=1)].index
        for idx in index_na:
            lat = X.loc[idx, self.lat_col]
            lon = X.loc[idx, self.lon_col]
            nbp = X.loc[idx, self.pieces_col]
            typ = X.loc[idx, self.type_col]
            neighbours = self.df_train_[
                (self.df_train_[self.pieces_col] == nbp) &
                (self.df_train_[self.type_col] == typ)
            ]
            distances = (lat - neighbours[self.lat_col])**2 + (lon - neighbours[self.lon_col])**2
            for col in self.quali_cols_:
                if pd.isna(X.loc[idx, col]):
                    valid = neighbours[neighbours[col].notna()]
                    if valid.shape[0] > 0:
                        idx_neigh = distances[valid.index].sort_values().iloc[:self.k].index
                        X.loc[idx, col] = neighbours.loc[idx_neigh][col].mode()[0]
                    else:
                        X.loc[idx, col] = self.default_mode_[col]
        
        return X

# Test des fonctions sur un découpage Train /test

In [4]:
df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')
print(df_ventes.shape)
df_ventes.head(2)


(27737, 58)


,type_annonceur,typedebien,typedetransaction,etage,surface,surface_terrain,nb_pieces,prix_bien,prix_maison,prix_terrain,...,UU2010,REG,DEP,loyer_m2_median_n6,nb_log_n6,taux_rendement_n6,loyer_m2_median_n7,nb_log_n7,taux_rendement_n7,prix_m2_vente
idannonce,,,,,,,,,,,,,,,,,,,,,
hektor-robischung-2862,pr,a,v,0,64,NaN,3,128400,NaN,NaN,...,68113,44,68,NaN,NaN,NaN,NaN,NaN,NaN,2006.25
apimo-82452515,pr,m,v,0,85,4950.0,4,189000,NaN,NaN,...,68000,44,68,NaN,NaN,NaN,NaN,NaN,NaN,2223.53


In [5]:
df_ventes_cleaned = clean_df(df_ventes)
print(df_ventes_cleaned.shape)
df_ventes_cleaned.head(2)

(27736, 48)


,type_annonceur,typedebien,typedetransaction,etage,surface,surface_terrain,nb_pieces,prix_bien,mensualiteFinance,balcon,...,TYP_IRIS_x,TYP_IRIS_y,GRD_QUART,UU2010,REG,DEP,loyer_m2_median_n7,nb_log_n7,taux_rendement_n7,prix_m2_vente
idannonce,,,,,,,,,,,,,,,,,,,,,
hektor-robischung-2862,pr,a,v,0,64,NaN,3,128400,0,1,...,Z,Z,6832500,68113,44,68,NaN,<NA>,NaN,2006.25
apimo-82452515,pr,m,v,0,85,4950.0,4,189000,0,0,...,Z,Z,6804400,68000,44,68,NaN,<NA>,NaN,2223.53


In [9]:
from sklearn.model_selection import train_test_split

target_feature = 'prix_bien'
target = df_ventes_cleaned[target_feature]
data = df_ventes_cleaned.drop(target_feature, axis=1)
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=66) 

In [10]:
print(X_train.shape, X_test.shape)

(22188, 47) (5548, 47)


### Visualisation des colonnes à valeurs manquantes de X_train et X_test pour comparer avant/après

In [15]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22188 entries, immo-facile-27543984 to immo-facile-35294998
Data columns (total 47 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           22188 non-null  object 
 1   typedebien               22188 non-null  object 
 2   typedetransaction        22188 non-null  object 
 3   etage                    22188 non-null  int64  
 4   surface                  22188 non-null  int64  
 5   surface_terrain          9158 non-null   float64
 6   nb_pieces                22188 non-null  int64  
 7   mensualiteFinance        22188 non-null  int64  
 8   balcon                   22188 non-null  int64  
 9   eau                      22188 non-null  int64  
 10  bain                     22188 non-null  int64  
 11  dpeL                     22188 non-null  object 
 12  dpeC                     13771 non-null  float64
 13  mapCoordonneesLatitude   22188 non-null  float6

In [16]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5548 entries, ag681494-377772445 to nexity-GB00131539
Data columns (total 47 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           5548 non-null   object 
 1   typedebien               5548 non-null   object 
 2   typedetransaction        5548 non-null   object 
 3   etage                    5548 non-null   int64  
 4   surface                  5548 non-null   int64  
 5   surface_terrain          2286 non-null   float64
 6   nb_pieces                5548 non-null   int64  
 7   mensualiteFinance        5548 non-null   int64  
 8   balcon                   5548 non-null   int64  
 9   eau                      5548 non-null   int64  
 10  bain                     5548 non-null   int64  
 11  dpeL                     5548 non-null   object 
 12  dpeC                     3379 non-null   float64
 13  mapCoordonneesLatitude   5548 non-null   float64
 14 

In [17]:
knn_imputer_custom = KNNImputerCustom()
X_train_transformed = knn_imputer_custom.fit_transform(X_train)
X_test_transformed = knn_imputer_custom.transform(X_test)

In [18]:
print(X_train_transformed.shape, X_test_transformed.shape)

(22188, 47) (5548, 47)


In [19]:
X_train_transformed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22188 entries, immo-facile-27543984 to immo-facile-35294998
Data columns (total 47 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           22188 non-null  object 
 1   typedebien               22188 non-null  object 
 2   typedetransaction        22188 non-null  object 
 3   etage                    22188 non-null  int64  
 4   surface                  22188 non-null  int64  
 5   surface_terrain          22188 non-null  float64
 6   nb_pieces                22188 non-null  int64  
 7   mensualiteFinance        22188 non-null  int64  
 8   balcon                   22188 non-null  int64  
 9   eau                      22188 non-null  int64  
 10  bain                     22188 non-null  int64  
 11  dpeL                     22188 non-null  object 
 12  dpeC                     22188 non-null  float64
 13  mapCoordonneesLatitude   22188 non-null  float6

In [20]:
X_test_transformed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5548 entries, ag681494-377772445 to nexity-GB00131539
Data columns (total 47 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           5548 non-null   object 
 1   typedebien               5548 non-null   object 
 2   typedetransaction        5548 non-null   object 
 3   etage                    5548 non-null   int64  
 4   surface                  5548 non-null   int64  
 5   surface_terrain          5548 non-null   float64
 6   nb_pieces                5548 non-null   int64  
 7   mensualiteFinance        5548 non-null   int64  
 8   balcon                   5548 non-null   int64  
 9   eau                      5548 non-null   int64  
 10  bain                     5548 non-null   int64  
 11  dpeL                     5548 non-null   object 
 12  dpeC                     5548 non-null   float64
 13  mapCoordonneesLatitude   5548 non-null   float64
 14 

### Test avec un échantillon et des valeurs par défaut dans le X_train (pour vérifier qu'elles sont bien prises dans le X_test_transformed)

In [21]:
df_ventes_sampled = df_ventes_cleaned[:1000]

In [22]:
target_feature = 'prix_bien'
target = df_ventes_sampled[target_feature]
data = df_ventes_sampled.drop(target_feature, axis=1)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=66) 

In [24]:
X_train["ascenseur"] = 10
X_train['chauffage_energie'] = 'Value_chauffage_energie'

In [25]:
index_test_asc = X_test[X_test['ascenseur'].isna()].index

In [26]:
index_test_chauff = X_test[X_test['chauffage_energie'].isna()].index

In [27]:
print(f"{X_test.shape[0]}, {len(index_test_asc)}, {len(index_test_chauff)}") 

200, 160, 127


In [28]:
knn_imputer_custom = KNNImputerCustom()
X_train_transformed = knn_imputer_custom.fit_transform(X_train)
X_test_transformed = knn_imputer_custom.transform(X_test)

In [29]:
X_test_transformed.loc[index_test_asc]['ascenseur'].value_counts()

ascenseur
10    160
Name: count, dtype: Int64

In [30]:
X_test_transformed.loc[index_test_chauff]['chauffage_energie'].value_counts()

chauffage_energie
Value_chauffage_energie    127
Name: count, dtype: int64